# KarateClub Node Classification with GNNVisualizer

This notebook trains four PyTorch Geometric node classifiers on the KarateClub dataset, then renders each trained model with `GNNVisualizer`. All four models use two message-passing layers and an 8-dimensional hidden feature width:

- GCN (`GCNConv`)
- GAT (`GATConv`)
- GraphSAGE (`SAGEConv`)
- GIN (`GINConv`)


If the PyG imports fail in a fresh notebook kernel, install the runtime packages in that environment first:

```bash
python3 -m pip install torch torch-geometric
```


In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GATConv, GCNConv, GINConv, SAGEConv

from gnn_exp import GNNVisualizer


In [2]:
SEED = 7
torch.manual_seed(SEED)

dataset = KarateClub()
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes
holdout_mask = ~data.train_mask

display(Markdown(
    f"KarateClub has **{data.num_nodes} nodes**, "
    f"**{data.edge_index.size(1)} directed edges**, "
    f"**{num_features} input features**, and **{num_classes} classes**. "
    f"PyG marks **{int(data.train_mask.sum())} labeled training nodes**; "
    f"the remaining **{int(holdout_mask.sum())} nodes** are used as a lightweight holdout set."
))


KarateClub has **34 nodes**, **156 directed edges**, **34 input features**, and **4 classes**. PyG marks **4 labeled training nodes**; the remaining **30 nodes** are used as a lightweight holdout set.

In [3]:
class GCNNodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.softmax(self.classifier(h))


class GATNodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("GAT hidden_channels must be divisible by 2")
        gat_heads = 2
        per_head_channels = hidden_channels // gat_heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=gat_heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.softmax(self.classifier(h))


class GraphSAGENodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.softmax(self.classifier(h))


def make_gin_mlp(in_channels, hidden_channels):
    return nn.Sequential(
        nn.Linear(in_channels, hidden_channels),
        nn.Tanh(),
        nn.Linear(hidden_channels, hidden_channels),
    )


class GINNodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(make_gin_mlp(in_channels, hidden_channels))
        self.act1 = nn.Tanh()
        self.conv2 = GINConv(make_gin_mlp(hidden_channels, hidden_channels))
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        return self.softmax(self.classifier(h))


In [4]:
def accuracy_for_mask(logits, labels, mask):
    if int(mask.sum()) == 0:
        return float("nan")
    pred = logits.argmax(dim=1)
    return float((pred[mask] == labels[mask]).float().mean())


def evaluate_model(model, graph_data):
    model.eval()
    with torch.no_grad():
        probs = model(graph_data.x, graph_data.edge_index)
    return {
        "train_acc": accuracy_for_mask(probs, graph_data.y, graph_data.train_mask),
        "holdout_acc": accuracy_for_mask(probs, graph_data.y, holdout_mask),
        "full_acc": accuracy_for_mask(probs, graph_data.y, torch.ones_like(graph_data.y, dtype=torch.bool)),
    }


def train_model(model, graph_data, epochs=300, lr=0.02, weight_decay=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    losses = []

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        probs = model(graph_data.x, graph_data.edge_index)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-9))[graph_data.train_mask], graph_data.y[graph_data.train_mask])
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))

    metrics = evaluate_model(model, graph_data)
    metrics["final_loss"] = losses[-1]
    return metrics


def display_metric_table(metrics_by_model):
    lines = [
        "| Model | Final loss | Train accuracy | Holdout accuracy | Full accuracy |",
        "|---|---:|---:|---:|---:|",
    ]
    for name, metrics in metrics_by_model.items():
        lines.append(
            f"| {name} | {metrics['final_loss']:.4f} | "
            f"{metrics['train_acc']:.2f} | {metrics['holdout_acc']:.2f} | {metrics['full_acc']:.2f} |"
        )
    display(Markdown("\n".join(lines)))


In [5]:
HIDDEN_CHANNELS = 8
MESSAGE_PASSING_LAYERS = 2

model_builders = {
    "GCN": lambda: GCNNodeClassifier(num_features, HIDDEN_CHANNELS, num_classes),
    "GAT": lambda: GATNodeClassifier(num_features, HIDDEN_CHANNELS, num_classes),
    "GraphSAGE": lambda: GraphSAGENodeClassifier(num_features, HIDDEN_CHANNELS, num_classes),
    "GIN": lambda: GINNodeClassifier(num_features, HIDDEN_CHANNELS, num_classes),
}

models = {}
metrics_by_model = {}

for name, build_model in model_builders.items():
    torch.manual_seed(SEED)
    model = build_model()
    metrics_by_model[name] = train_model(model, data)
    models[name] = model.eval()

display_metric_table(metrics_by_model)


| Model | Final loss | Train accuracy | Holdout accuracy | Full accuracy |
|---|---:|---:|---:|---:|
| GCN | 0.0042 | 1.00 | 0.67 | 0.71 |
| GAT | 0.0035 | 1.00 | 0.67 | 0.71 |
| GraphSAGE | 0.0023 | 1.00 | 0.43 | 0.50 |
| GIN | 0.0025 | 1.00 | 0.63 | 0.68 |

The next cells build one `GNNVisualizer` per trained model. The query pair `[0, 33]` highlights two opposite endpoints in the KarateClub graph so the same graph region is easy to compare across architectures.

In [6]:
QUERY_PAIR = [0, 33]
EXPECTED_LAYER_TYPES = {
    "GCN": "GCNConv",
    "GAT": "GATConv",
    "GraphSAGE": "SAGEConv",
    "GIN": "GINConv",
}


def make_visualizer(model, query_pair):
    visualizer = GNNVisualizer(renderer="svg")
    visualizer.add_model(
        data=data,
        model=model,
        subgraphSample=False,
        queries=[query_pair],
        mode="node",
    )
    return visualizer


visualizers = {
    name: make_visualizer(model, QUERY_PAIR)
    for name, model in models.items()
}

summary_rows = [
    "| Model | First layer | Aggregation | Message-passing layers | Hidden width | Stored nodes | Query |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for name, visualizer in visualizers.items():
    first_layer = visualizer.modelInfo["conv1"]
    assert first_layer["type"] == EXPECTED_LAYER_TYPES[name]
    assert len(visualizer.graphData["x"]) == data.num_nodes
    assert len(visualizer.intmData["act0"]) == data.num_nodes
    assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS
    assert len(visualizer.intmData["act2"][0]) == HIDDEN_CHANNELS
    message_layers = sum(
        1 for layer in visualizer.modelInfo.values()
        if layer.get("type") in EXPECTED_LAYER_TYPES.values()
    )
    assert message_layers == MESSAGE_PASSING_LAYERS
    summary_rows.append(
        f"| {name} | `{first_layer['type']}` | `{first_layer.get('aggregation')}` | "
        f"{message_layers} | {len(visualizer.intmData['act1'][0])} | "
        f"{len(visualizer.graphData['x'])} | `{visualizer.queries}` |"
    )

display(Markdown("\n".join(summary_rows)))


mode: node
Python: queries changed to: [[0, 33]], type: <class 'list'>
Queries set to: [[0, 33]]
check act0: [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
modelInfo: dict_keys(['conv1',

| Model | First layer | Aggregation | Message-passing layers | Hidden width | Stored nodes | Query |
|---|---:|---:|---:|---:|---:|---:|
| GCN | `GCNConv` | `gcn-normalized` | 2 | 8 | 34 | `[[0, 33]]` |
| GAT | `GATConv` | `attention` | 2 | 8 | 34 | `[[0, 33]]` |
| GraphSAGE | `SAGEConv` | `mean` | 2 | 8 | 34 | `[[0, 33]]` |
| GIN | `GINConv` | `sum` | 2 | 8 | 34 | `[[0, 33]]` |

## GCN

In [7]:
display(visualizers["GCN"])


## GAT

In [8]:
display(visualizers["GAT"])


## GraphSAGE

In [9]:
display(visualizers["GraphSAGE"])


## GIN

In [10]:
display(visualizers["GIN"])
